# Un problema de árboles de decisión

En este taller estudiarás conceptos mostrados en el tutorial "Solución de problemas con árboles de decisión", utilizando el conjunto de datos correspondiente al desempeño y desgaste de empleados. Particularmente, realizarás los siguientes procesos:

1. Cargar un conjunto de datos.
2. Preparar los datos para el modelado.
3. Realizar la búsqueda de hiperparámetros para el modelo de árbol de decisión.
4. Evaluar el mejor modelo resultante.

Entonces, dadas algunas características en áreas como la educación, trabajos previos, salario, entre otras, queremos clasificar a un empleado como agotado (con bajo desempeño) o no. Antes de iniciar, vamos a importar las librerías necesarias:

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay
from sklearn.preprocessing import OneHotEncoder

from importlib.metadata import version

print(f"Versión de Pandas: {version('pandas')}")
print(f"Versión de Matplotlib: {version('matplotlib')}")
print(f"Versión de Scikit-learn: {version('scikit-learn')}")
print(f"Versión de Numpy: {version('numpy')}")

Versión de Pandas: 2.2.3
Versión de Matplotlib: 3.10.0
Versión de Scikit-learn: 1.6.1
Versión de Numpy: 1.25.2


## 1. Carga de datos

Con las librerías importadas, realizaremos la carga del conjunto de datos:

### Ejercicio 1.1.

Utiliza Pandas para importar el archivo que contiene el conjunto de datos de desgaste de empleados.

* La ruta del archivo .csv es: `./data/EmployeeAttrition.csv`, y ya se encuentra en el entorno de Coursera, solo debes importarlo.
* La variable resultante debe tener el nombre `data_raw`, que representa el conjunto de datos sin modificar.

In [ ]:
ruta = './data/Employee-Attrition.csv'
# your code here


In [ ]:
#---------- Celda de Pruebas ----------
# El resultado existe
# El resultado es un DataFrame
# El resultado tiene las dimensiones correctas
#--------------------------------------

# Se verifica que la variable exista
assert data_raw is not None, "Asegúrate de definir la variable /`data_raw/` igualándola a una función de Pandas para leer el archivo."

# Se verifica que sea un DataFrame
assert isinstance(data_raw, pd.DataFrame), "El resultado debe ser un DataFrame."

# Se evalúan las dimensiones de la variable
assert data_raw.shape == (1470,32), "¿Verificaste que la ruta del archivo CSV y el nombre de la variable son correctos?"
print("¡Los datos tienen las dimensiones correctas!")

En este punto es conveniente observar la estructura de los datos. ¿Cuántas variables categóricas tiene este conjunto de datos?

## 2. Preparación de datos

Primero vamos a definir la variable `data` para almacenar un conjunto de datos modificado:

In [ ]:
data = data_raw.copy()

A continuación, verificaremos si hay valores faltantes:

In [ ]:
data.isna().sum()

Y verificaremos si hay filas duplicadas:

In [ ]:
data.duplicated().sum()

Como puedes observar, no tenemos que hacer una limpieza previa, por lo que seguiremos con la preparación de los datos.

### Ejercicio 2.1.

Ahora debes dividir el conjunto de datos en entrenamiento y pruebas. Usando el 80% de los datos para entrenar el modelo y el 20% restante para probarlo, utiliza `scikit-learn` para separar el conjunto de datos en dos.

* Guarda tu respuesta en dos variables: `train` y `test`. (**Ejemplo: `train, test = <<Función>>`**)
* Utiliza el parámetro `random_state=0`. Esto hará que la partición sea siempre la misma.
* Encontrarás la línea `train.head()`. Esta línea se usa para que puedas visualizar el resultado del conjunto de entrenamiento. Déjala al final de la celda y no la modifiques.

In [ ]:
# your code here

train.head()

In [ ]:
#---------- Celda de Pruebas ----------
# Las variables "train" y "test" existen
# Las variables "train" y "test" son un DataFrame
# Las variables tienen las dimensiones correctas
#--------------------------------------

# Se verifica que "train" y "test" están definidas
assert train is not None, "Asegúrate de definir la variable \'train\' con el nombre correcto."
assert test is not None, "Asegúrate de definir la variable \'test\' con el nombre correcto."

# Se verifica que "train" y "test" sean un DataFrame
assert isinstance(train, pd.DataFrame), "La variable \'train\' debe ser un DataFrame."
assert isinstance(test, pd.DataFrame), "La variable \'test\' debe ser un DataFrame."

# Se evalúan las dimensiones de las variables
assert train.shape == (1176,32), "Verifica que estés utilizando el 80% de los datos para el conjunto de entrenamiento."
assert test.shape == (294,32), "Verifica que estés utilizando el 20% de los datos para el conjunto de pruebas."
print("¡Los conjuntos de entrenamiento y pruebas tienen las dimensiones correctas!")

### Ejercicio 2.2.

Ahora debes aislar la variable objetivo, `Attrition`, de las variables independientes. Utiliza Pandas para crear dos variables, `x_train` y `y_train`, que almacenarán las variables independientes y la variable objetivo, respectivamente.

* Crea una variable con nombre `x_train` y asígnale la operación necesaria para almacenar solo las variables independientes del conjunto de entrenamiento. (**Ejemplo: `x_train = train.<<Función>>`**)
* Crea una variable con nombre `y_train` y asígnale la operación necesaria para almacenar la variable objetivo del conjunto de entrenamiento. (**Ejemplo: `y_train = <<Consulta>>`**)

In [ ]:
# Tu respuesta deben ser dos líneas consecutivas:
#    x_train = train.<<Función>>
#    y_train = <<Consulta>>
# your code here


In [ ]:
#---------- Celda de Pruebas ----------
# Las variables "x_train" y "y_train" existen
# La variable "x_train" es un DataFrame
# La variable "y_train" es una Serie de Pandas
# Las variables tienen las dimensiones correctas
#--------------------------------------

# Se verifica que las variables están definidas
assert x_train is not None, "Asegúrate de definir la variable \'x_train\' correctamente."
assert y_train is not None, "Asegúrate de definir la variable \'y_train\' correctamente."

# Se verifica que "x_train" sea un DataFrame
assert isinstance(x_train, pd.DataFrame), "El resultado debe ser un DataFrame."

# Se verifica que "y_train" sea una Serie
assert isinstance(y_train, pd.Series), "El resultado debe ser una Serie de Pandas."

# Se evalúan las dimensiones de las variables
assert x_train.shape == (1176,31), "\'x_train\' debe tener el mismo número de filas pero una columna menos que \'train\'."
assert y_train.shape == (1176,), "\'y_train\' solamente contiene una columna."
print("¡Los conjuntos \'x_train\' y \'y_train\' tienen las dimensiones correctas!")

### Ejercicio 2.3.

Ahora vas a realizar la transformación de las variables categóricas. Utiliza el encoder definido junto con las variables categóricas y numéricas para transformar todas las variables categóricas en variables numéricas que puedan ser interpretadas por nuestro modelo.


Aquí definimos las variables categóricas y numéricas 

In [ ]:
vars_cat= list(x_train.select_dtypes(include=["object",bool], exclude=[int,float]).columns)
vars_num=list(set(x_train.columns)-set(vars_cat))
print(f"variables categóricas: {vars_cat}")
print(f"variables numéricas: {vars_num}")

En este paso definimos el OneHotEncoder

In [ ]:
encoder = OneHotEncoder(sparse_output=False).set_output(transform = "pandas")

En esta parte separamos las variables categóricas de las numéricas

In [ ]:
x_train_cat = x_train[vars_cat]
x_train_num = x_train[vars_num]

* Utiliza el encoder para **ajustarlo** a las variables categóricas existentes (Ejemplo: **encoder.<<Función>>**)
* Utiliza el encoder para **transformar** las variables categóricas a numéricas (Ejemplo: **x_train_cat = encoder.<<Función>>**)
* Combina nuevamente las variables categóricas y numéricas en un único dataset y guárdalo en la variable `x_train`. (Ejemplo: **pd.concat(<<Parámetros>>)**)
* Encontrarás la línea `x_train.head()` al final de la celda. Esta línea se usa para que puedas visualizar el resultado de la transformación, por lo que debes dejarla al final y no debes modificarla.

In [ ]:
# your code here

x_train.head()

In [ ]:
#---------- Celda de Pruebas ----------
# La variable 'x_train' existe
# La variable 'x_train' es un DataFrame
# La variable 'x_train' ha aumentado su número de columnas
#--------------------------------------

# Se verifica que la variable exista
assert x_train is not None, "Asegúrate de definir la variable /`data/` igualándola a una función de Pandas."

# Se verifica que sea un DataFrame
assert isinstance(x_train, pd.DataFrame), "El resultado debe ser un DataFrame."

# Se evalúan las dimensiones de la variable data
assert x_train.shape == (1176,52), "Recuerda que la codificación de variables categóricas resulta en un aumento del número de columnas."
print("¡Los datos tienen las dimensiones correctas!")

Debido a que los árboles de decisión no necesitan estandarización, podemos pasar directamente a la búsqueda de hiperparámetros.

## 3. Búsqueda de hiperparámetros

Con el conjunto de entrenamiento preparado, realizaremos una búsqueda de los hiperparámetros `criterion`, `max_depth` y `min_samples_split`. Para esto utilizaremos un objeto de la clase `KFold`, con el que definiremos 10 subconjuntos sobre el conjunto de entrenamiento:

In [ ]:
kfold = KFold(n_splits=10, shuffle=True, random_state=0)

Además, definiremos el objeto de la clase `DecisionTreeClassifier` para realizar el entrenamiento con un árbol de decisión. Ten en cuenta que usamos el parámetro `random_state=0` para que, después de definir una división, la reorganización de las variables restantes sea siempre la misma:

In [ ]:
decision_tree = DecisionTreeClassifier(random_state=0)

### Ejercicio 3.1.

El siguiente paso es definir el espacio de búsqueda de los hiperparámetros.

* Define una variable con el nombre `param_grid` y asígnale la expresión necesaria para crear un diccionario con tres tuplas (**Ejemplo: `param_grid = <<Expresión>>`**):
    * Llave `criterion` y valor `valores_criterion`.
    * Llave `max_depth` y valor `valores_max_depth`.
    * Llave `min_samples_split` y valor `valores_min_samples_split`.

In [ ]:
valores_criterion = ['entropy', 'gini']
valores_max_depth = [2, 4, 6, 8]
valores_min_samples_split = [2, 3, 5, 10]
# your code here


In [ ]:
#---------- Celda de Pruebas ----------
# La variable "param_grid" existe
# La variable "param_grid" es un diccionario
# La variable tiene la longitud correcta
#--------------------------------------

# Se verifica que "param_grid" está definida
assert param_grid is not None, "Asegúrate de definir la variable \'param_grid\' correctamente."

# Se verifica que "param_grid" sea un diccionario
assert isinstance(param_grid, dict), "El resultado debe ser un diccionario de Python, es decir, no necesitas ninguna función específica de ninguna librería para definir la variable."

# Se evalúa la longitud de la variable
assert len(param_grid) == 3, "\'param_grid\' debe contener tres tuplas."
print("¡\'param_grid\' está definida correctamente!")

### Ejercicio 3.2.

Finalmente, el último paso antes de realizar la búsqueda de hiperparámetros es crear el objeto de tipo `GridSearchCV`. Utiliza las variables `decision_tree`, `param_grid` y `kfold` para definirlo:

* Define una variable con el nombre `grid` y asígnale la función necesaria para crear un objeto de la clase `GridSearchCV`. (**Ejemplo: `grid = <<Función>>`**)
* Utiliza el parámetro `scoring='accuracy'` para que se seleccione el mejor modelo de acuerdo con los valores de exactitud.

In [ ]:
# your code here


In [ ]:
#---------- Celda de Pruebas ----------
# La variable "grid" existe
# La variable "grid" es un objeto de la clase GridSearchCV
# La variable "grid" usa la exactitud como método de selección
#--------------------------------------

# Se verifica que "grid" está definida
assert grid is not None, "Asegúrate de definir la variable \'grid\' correctamente."

# Se verifica que "grid" es un objeto de la clase GridSearchCV
assert type(grid) == GridSearchCV, "La variable \'grid\' debe ser un objeto de la clase GridSearchCV."

# Se verifica que "grid" tiene un atributo "scoring" igual a "accuracy"
assert grid.scoring == 'accuracy', "La variable \'grid\' debe usar la exactitud como método de selección."
print("¡\'grid\' está definida correctamente!")

### Ejercicio 3.3.

A continuación, realiza la búsqueda de hiperparámetros utilizando el conjunto de entrenamiento, compuesto por las variables `x_train` y `y_train`.

* Para este ejercicio no debes asignar tu resultado a ninguna variable. Es decir, solo debes ejecutar una función sobre la variable `grid`, utilizando las variables `x_train` y `y_train` como parámetros. (**Ejemplo: `grid.<<Función>>`**)

In [ ]:
# your code here


In [ ]:
#---------- Celda de Pruebas ----------
# El atributo "best_params_" de la variable "grid" existe
# El atributo "best_estimator_" de la variable "grid" existe
#--------------------------------------

# El atributo "best_params_" está definido
assert grid.best_params_ is not None, "Asegúrate de ejecutar la función de entrenamiento para generar un diccionario con los mejores hiperparámetros."

# El atributo "best_estimator_" está definido
assert grid.best_estimator_ is not None, "Asegúrate de ejecutar la función de entrenamiento para generar un modelo de árboles de decisión."

# Se verifica que "grid" haya generado tres mejores hiperparámetros
assert len(grid.best_params_) == 3, "Al ejecutar una función usando el objeto \'grid\', se deben generar tres tuplas con los hiperparámetros."

# Se verifica que "grid" haya generado un modelo de árboles de decisión
assert isinstance(grid.best_estimator_, DecisionTreeClassifier), "Al ejecutar una función usando el objeto \'grid\', se debe generar un modelo de árboles de decisión."
print("¡Se ha realizado la búsqueda de hiperparámetros correctamente!")

Ahora obtendremos los mejores valores de los hiperparámetros usando `grid.best_params_`:

In [ ]:
print("Mejores parámetros: {}".format(grid.best_params_)) 

Además, almacenaremos el mejor modelo utilizando `grid.best_estimator_`:

In [ ]:
mejor_modelo = grid.best_estimator_

### Ejercicio 3.4.

Utilizando el mejor modelo, obtén el arreglo de importancia de las variables utilizadas para los criterios de división.

* Para este ejercicio define una variable `importancia` que debe contener un arreglo de valores representando la importancia de cada variable del mejor modelo de árbol de decisión. (**Ejemplo: `importancia = mejor_modelo.<<atributo>>`**)
* Encontrarás dos líneas para convertir el arreglo en un DataFrame y para ordenarlo. Debes dejar estas líneas al final de la celda y no las debes modificar.

In [ ]:
# your code here

importancia_df = pd.DataFrame(data={'Variable':x_train.columns, 'Importancia':importancia})
importancia_df.sort_values('Importancia', ascending=False)

In [ ]:
#---------- Celda de Pruebas ----------
# La variable "importancia" existe
# La variable "importancia" es un arreglo de numpy
# La variable "importancia" tiene las dimensiones correctas
#--------------------------------------

# Se verifica que "importancia" está definida
assert importancia is not None, "Asegúrate de definir la variable \'importancia\' correctamente."

# Se verifica que "importancia" es un arreglo de numpy
assert type(importancia) == np.ndarray, "La variable \'importancia\' debe ser un arreglo de numpy."

# Se verifica que "importancia" tiene las dimensiones correctas
assert importancia.shape == (52,), "La variable \'importancia\' debe tener dimensiones (52,)."
print("¡\'importancia\' está definida correctamente!")

¿Qué variables son las más relevantes para nuestro árbol? ¿Qué variables no son utilizadas? A continuación puedes observar el árbol resultante para visualizar las importancias obtenidas:

In [ ]:
plt.figure(figsize=(25,10))
tree = plot_tree(mejor_modelo, feature_names=x_train.columns, class_names=['No','Yes'], filled=True, fontsize=9)

## 4. Evaluación del modelo

Por último, evaluarás el modelo entrenado utilizando el conjunto de prueba.

### Ejercicio 4.0.

Separa las variables independientes y la variable objetivo en el conjunto de pruebas. Para ello, utiliza Pandas para crear dos variables, `x_test` y `y_test`, que almacenarán las variables independientes y la variable objetivo, respectivamente.

* Crea una variable con nombre `x_test` y asígnale la operación necesaria para almacenar solo las variables independientes del conjunto de pruebas. (**Ejemplo: `x_test = test.<<Función>>`**)
* Crea una variable con nombre `y_test` y asígnale la operación necesaria para almacenar la variable objetivo del conjunto de pruebas. (**Ejemplo: `y_test = <<Consulta>>`**)

In [ ]:
# Tu respuesta deben ser dos líneas consecutivas:
#    x_test = test.<<Función>>
#    y_test = <<Consulta>>
# your code here


In [ ]:
#---------- Celda de Pruebas ----------
# Las variables existen
# La variable "x_test" es un DataFrame
# La variable "y_test" es una Serie de Pandas
# Las variables tienen las dimensiones correctas
#--------------------------------------

# Se verifica que las variables están definidas
assert x_test is not None, "Asegúrate de definir la variable con el nombre correcto."
assert y_test is not None, "Asegúrate de definir la variable con el nombre correcto."

# Se verifica que "x_test" sea un DataFrame
assert isinstance(x_test, pd.DataFrame), "El resultado debe ser un DataFrame."

# Se verifica que "y_test" sea una Serie
assert isinstance(y_test, pd.Series), "El resultado debe ser una Serie de Pandas."

# Se evalúan las dimensiones de las variables
assert x_test.shape == (294,31), "\'x_test\' debe tener el mismo número de filas pero una columna menos que \'test\'."
assert y_test.shape == (294,), "\'y_test\' solamente contiene una columna."
print("¡Los conjuntos \'x_test\' y \'y_test\' tienen las dimensiones correctas!")

### Ejercicio 4.1.

Realiza la transformación de las variables categóricas. Utiliza el encoder definido junto con las variables categóricas y numéricas para transformar todas las variables categóricas en variables numéricas que puedan ser interpretadas por nuestro modelo.

1. Separa los datos de test en categóricos (con las columnas de la variable `vars_cat`) y numéricos (con las columnas de la variable `vars_num`) usando pandas
2. Codifica los datos categóricos usando el encoder (Ejemplo: `categoricos_encoded = encoder.<<Función>>`)
3. Junta nuevamente los datos numéricos y categóricos en un único conjunto de test, y guárdalos en `x_test`, sobreescribiendo su contenido (Ejemplo: `x_test = pd.<<Función>>`)

In [ ]:
# your code here

x_test.head()

In [ ]:
#---------- Celda de Pruebas ----------
# La variable "x_test" es un DataFrame
# La variable tiene las dimensiones correctas
#--------------------------------------

# Se verifica que "x_test" sea un DataFrame
assert isinstance(x_test, pd.DataFrame), "El resultado debe ser un DataFrame."

# Se evalúan las dimensiones de la variable
assert x_test.shape == (294,52), "\'x_test\' debe mantener las mismas dimensiones, solo se modifican sus valores."
print("¡\'x_test\' es un DataFrame con las dimensiones correctas!")

### Ejercicio 4.2.

Con el conjunto de pruebas preparado, realiza predicciones con el fin de compararlas con los valores reales almacenados en `y_test`.

* Utiliza la variable `mejor_modelo` para realizar las predicciones sobre el mejor modelo. Asigna el resultado a una variable con nombre `y_pred` (**Ejemplo: `y_pred = mejor_modelo.<<Función>>`**).

In [ ]:
# your code here


In [ ]:
#---------- Celda de Pruebas ----------
# La variable "y_pred" existe
# La variable "y_pred" es un arreglo
# La variable "y_pred" tiene las dimensiones correctas
#--------------------------------------

# Se verifica que "y_pred" está definida
assert y_pred is not None, "Asegúrate de definir la variable con el nombre correcto."

# Se verifica que "y_pred" sea un arreglo
assert isinstance(y_pred, np.ndarray), "El resultado debe ser un arreglo de Numpy."

# Se evalúan las dimensiones de "y_pred"
assert y_pred.shape == (294,), "\'y_pred\' debe tener el mismo número de filas (predicciones) que \'x_test\' y \'y_test\'."
print("¡\'y_pred\' es un arreglo con las dimensiones correctas!")

### Ejercicio 4.3.

Utiliza el conjunto de predicciones `y_pred` y el conjunto de valores reales `y_test` para obtener la matriz de confusión del mejor modelo.

* Haz un llamado a la función que retorna la matriz de confusión como un arreglo. Asigna el resultado a una nueva variable con el nombre `p43` (**Ejemplo: `p43 = <<Función>>`**).
* Encontrarás la línea `p43` al final de la celda. Esta línea se usa para que puedas visualizar la matriz resultante, por lo que no la debes modificar.

In [ ]:
# your code here

p43

In [ ]:
#---------- Celda de Pruebas ----------
# La variable "p43" existe
# La variable "p43" es un arreglo de numpy
# La variable "p43" tiene las dimensiones correctas
#--------------------------------------

# Se verifica que "p43" está definida
assert p43 is not None, "Asegúrate de definir la variable con el nombre correcto."

# Se verifica que "p43" sea un arreglo de numpy
assert isinstance(p43, np.ndarray), "El resultado debe ser un arreglo de Numpy."

# Se evalúan las dimensiones de "p43"
assert p43.shape == (2,2), "\'p43\' debe tener dos filas y dos columnas, equivalente al número de clases del problema."
print("¡\'p43\' es un arreglo con las dimensiones correctas!")

### Ejercicio 4.4.

Finalmente, obtén cuatro métricas de rendimiento utilizando las predicciones del modelo.

#### Ejercicio 4.4.1.

Utiliza `scikit-learn` para obtener la exactitud (accuracy).

* Define una variable con el nombre `accuracy` y asígnale la función necesaria para obtener la exactitud (`accuracy = <<Función>>`).
* Encontrarás una línea que mostrará tu resultado. Esta línea debe ir al final y no la debes modificar.

In [ ]:
# your code here

print("Exactitud: ", accuracy)

In [ ]:
#---------- Celda de Pruebas ----------
# La variable "accuracy" existe
# La variable "accuracy" es un número
# La variable "accuracy" tiene un valor válido
#--------------------------------------

# Se verifica que "accuracy" está definida
assert accuracy is not None, "Asegúrate de definir la variable con el nombre correcto."

# Se verifica que "accuracy" sea un número
assert isinstance(accuracy, np.float64) or isinstance(accuracy, float), "El resultado debe ser un número decimal."

# Se verifica que "accuracy" tiene un valor válido
assert accuracy >= 0 and accuracy <= 1, "\'accuracy\' debe tener un valor entre 0 y 1."
print("¡\'accuracy\' es un número válido!")

#### Ejercicio 4.4.2.

Utiliza `scikit-learn` para obtener la sensibilidad (recall).

* Define una variable con el nombre `recall` y asígnale la función necesaria para obtener la sensibilidad (`recall = <<Función>>`).
* Encontrarás una línea que mostrará tu resultado. Esta línea debe ir al final y no la debes modificar.

In [ ]:
# your code here

print("Sensibilidad: ", recall)

In [ ]:
#---------- Celda de Pruebas ----------
# La variable "recall" existe
# La variable "recall" es un número
# La variable "recall" tiene un valor válido
#--------------------------------------

# Se verifica que "recall" está definida
assert recall is not None, "Asegúrate de definir la variable con el nombre correcto."

# Se verifica que "recall" sea un número
assert isinstance(recall, np.float64) or isinstance(recall, float), "El resultado debe ser un número decimal."

# Se verifica que "recall" tiene un valor válido
assert recall >= 0 and recall <= 1, "\'recall\' debe tener un valor entre 0 y 1."
print("¡\'recall\' es un número válido!")

#### Ejercicio 4.4.3.

Utiliza `scikit-learn` para obtener la precisión.

* Define una variable con el nombre `precision` y asígnale la función necesaria para obtener la precisión (`precision = <<Función>>`).
* Encontrarás una línea que mostrará tu resultado. Esta línea debe ir al final y no la debes modificar.

In [ ]:
# your code here

print('Precisión: ', precision)

In [ ]:
#---------- Celda de Pruebas ----------
# La variable "precision" existe
# La variable "precision" es un número
# La variable "precision" tiene un valor válido
#--------------------------------------

# Se verifica que "precision" está definida
assert precision is not None, "Asegúrate de definir la variable con el nombre correcto."

# Se verifica que "precision" sea un número
assert isinstance(precision, np.float64) or isinstance(precision, float), "El resultado debe ser un número decimal."

# Se verifica que "precision" tiene un valor válido
assert precision >= 0 and precision <= 1, "\'precision\' debe tener un valor entre 0 y 1."
print("¡\'precision\' es un número válido!")

#### Ejercicio 4.4.4.

Utiliza `scikit-learn` para obtener el F1-Score.

* Define una variable con el nombre `f1` y asígnale la función necesaria para obtener el F1-Score (`f1 = <<Función>>`).
* Encontrarás una línea que mostrará tu resultado. Esta línea debe ir al final y no la debes modificar.

In [ ]:
# your code here

print('F1-Score: ', f1)

In [ ]:
#---------- Celda de Pruebas ----------
# La variable "f1_score" existe
# La variable "f1_score" es un número
# La variable "f1_score" tiene un valor válido
#--------------------------------------

# Se verifica que "f1_score" está definida
assert f1 is not None, "Asegúrate de definir la variable con el nombre correcto."

# Se verifica que "f1_score" sea un número
assert isinstance(f1, np.float64) or isinstance(f1, float), "El resultado debe ser un número decimal."

# Se verifica que "f1_score" tiene un valor válido
assert f1 >= 0 and f1 <= 1, "\'f1_score\' debe tener un valor entre 0 y 1."
print("¡\'f1_score\' es un número válido!")

## Cierre

Al realizar los ejercicios de este taller, has reforzado tus capacidades para realizar una búsqueda exhaustiva de hiperparámetros, visualizar la importancia de las variables y el árbol resultante, además de evaluar el modelo utilizando la matriz de confusión y cuatro métricas de rendimiento. 

---
Creado por: Nicolás Díaz

Última edición: Camilo Rozo

Revisado por: Haydemar Nuñez

Versión: Febrero 2025

Universidad de los Andes